In [ ]:
transactions_silver = spark.table("workspace.default.silver_transactions")
users_silver = spark.table("workspace.default.silver_users")
cards_silver = spark.table("workspace.default.silver_cards")

In [ ]:
print("transactions col count", len(transactions_silver.columns))
print("users col count", len(users_silver.columns))
print("cards col count", len(cards_silver.columns))

In [ ]:
transactions_silver.select("user_id").distinct().count()

In [ ]:
transactions_silver.select("user_id", "card_id", "merchant_id").show(5, False)

In [ ]:
users_silver.select("user_id", "customer_name", "customer_segment").show(5, False)

In [ ]:
cards_silver.select("user", "card_index", "card_brand", "card_type").show(5, False)

In [ ]:
cards_silver.select("user").distinct().count()

In [ ]:
cards_silver.select("user", "card_index").orderBy("user", "card_index").show(10)

In [ ]:
transactions_silver.printSchema()
users_silver.printSchema()
cards_silver.printSchema()

In [ ]:
transactions_silver.select("user_id","card_id").distinct().count()
transactions_silver.select("user_id","card_id").distinct().show(5)

In [ ]:
#Validating if all transactions_silver user_id and cards_id are present in cards_silver user_id and card_index as cards_id 

from pyspark.sql import functions as F
card_reference = (
    cards_silver
    .select(
        F.col("user").alias("user_id"),
        F.col("card_index").alias("card_id")
    )
    .distinct()
    .withColumn("card_exists", F.lit(1))
)

transaction_card_check = (
    transactions_silver
    .select("user_id", "card_id")
    .distinct()
    .join(
        card_reference,
        ["user_id", "card_id"],
        "left"
    )
)

transaction_card_check.filter(
    F.col("card_exists").isNull()
).count()

Creating dim_customers and validating it

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
dim_customers = users_silver.select(F.col("user_id").alias("customer_id"),"customer_name","age","gender","city","state","zip_code","per_capita_income","annual_income","total_debt","fico_score","num_credit_cards","income_band","debt_to_income_ratio","credit_profile", "customer_segment").withColumn("customer_key",F.row_number().over(Window.orderBy("customer_id"))).select( "customer_key","customer_id","customer_name","age","gender","city","state","zip_code","per_capita_income","annual_income","total_debt","fico_score","num_credit_cards","income_band","debt_to_income_ratio","credit_profile","customer_segment"
    )

In [ ]:
from pyspark.sql.functions import *
print("count of dim_customers",dim_customers.count())
print("Counting distinct customer_key",dim_customers.select("customer_key").distinct().count())
print("Counting distinct customer_id",dim_customers.select("customer_id").distinct().count())
print("Counting null values in customer_key",dim_customers.filter(
    F.col("customer_key").isNull()
).count())

Creating table dim_date

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
dim_date = transactions_silver.select("transaction_date","transaction_year", "transaction_month", "transaction_day","day_of_week").distinct().withColumnRenamed("transaction_date","date").withColumnRenamed("transaction_year","year").withColumnRenamed("transaction_month","month").withColumnRenamed("transaction_day","day").withColumn("date_key", F.date_format("date","yyyyMMdd").cast("int")).withColumn("month_name",F.date_format("date","MMMM")).withColumn("quarter",F.quarter("date")).withColumn("day_of_year",F.dayofyear("date")).withColumn("week_of_year",F.weekofyear("date")).select("date_key","date","year","quarter","month","month_name","day","day_of_week","day_of_year","week_of_year").orderBy("date")

In [ ]:
print("date count",dim_date.count())
print("Counting date key",dim_date.select("date_key").distinct().count())
print("Counting date",dim_date.select("date").distinct().count())

In [ ]:
dim_date.show(5,False)

Creating dim_cards

In [ ]:
dim_cards = cards_silver.select(col("user").alias("customer_id")).show(5)

In [ ]:
dim_cards = cards_silver.select(col("user").alias("customer_id"),col("card_index").alias("card_id"),"card_brand","card_type","has_chip","cards_issued","credit_limit","acct_open_date","cards_status", "card_age_years", "credit_limit_band"
    ).withColumn("card_key",sha2(concat_ws("_",col("customer_id").cast("string"),col("card_id").cast("string")),256)).select("card_key","customer_id","card_id","card_brand","card_type","has_chip","cards_issued","credit_limit","acct_open_date", "cards_status", "card_age_years", "credit_limit_band"
    )


In [ ]:
print("Count the dim cards",dim_cards.count())
print("Cound cards_key",dim_cards.select("card_key").count())
print("Count nulls in cards_key", dim_cards.filter(col("card_key").isNull()).count())

Creating dim_merchant

In [ ]:
merchant_check = transactions_silver.groupBy("merchant_id").agg(countDistinct(struct("merchant_city","merchant_state","mcc")).alias("merchant_combination"))

In [ ]:
merchant_check.filter(
    F.col("merchant_combination") > 1
).count()

In [ ]:
merchant_check.filter(F.col("merchant_combination")>1).orderBy(F.desc("merchant_combination")).show(10,False)

In [ ]:
transactions_silver.filter(F.col("merchant_id")== -4282466774399734331).select("merchant_id","merchant_city","merchant_state","mcc").distinct().show()

In [ ]:
#Seeing above results decided to make merchant_key as a combination of merchant_id,merchant_city, merchant_starte, mcc
dim_merchant = transactions_silver.select(col("merchant_id"),col("merchant_city"),col("merchant_state"),col("mcc")).distinct().withColumn("merchant_key",F.sha2(concat_ws("_", col("merchant_id").cast("string"),coalesce(col("merchant_city"),lit("UNKNOWN")),coalesce(col("merchant_state"),lit("UNKNOWN")),coalesce(col("mcc"),lit("UNKNOWN"))), 256)).select("merchant_key","merchant_id","merchant_city","merchant_state","mcc")

In [ ]:
print("count dim_merchnat()", dim_merchant.count())
print("count merchant_ket",dim_merchant.select("merchant_key").distinct().count())
print("count null merchant_key",dim_merchant.filter(F.col("merchant_key").isNull()).count())
print("count merchant_key by grouping",dim_merchant.groupBy("merchant_key").agg(count("*").alias("count")).filter(F.col("count")>1).count())

Creating fact_base table

In [ ]:
fact_base = transactions_silver.select("transaction_id","user_id","card_id","transaction_date","merchant_id","merchant_city","merchant_state","mcc","amount","payment_method",F.col("transaction_Status").alias("transaction_status"),"fraud_flag")


In [ ]:
print("count fact_base",fact_base.count())
print("count distinct fact_base transaction_id",fact_base.select("transaction_id").distinct().count())

Building and Validating Fact Table

In [ ]:
#Adding customer_key as foreign key in fact table
fact_with_customer = fact_base.alias("f").join(dim_customers.select("customer_id","customer_key").alias("c"),F.col("user_id")==F.col("customer_id"),how="left").select("f.*",F.col("c.customer_key")
    )

In [ ]:
#Validating the counts for customer_key
print("count fact_with_customer", fact_with_customer.count())
print("counting null for customer_key", fact_with_customer.filter(F.col("customer_key").isNull()).count())

In [ ]:
#Adding card key as foreign key in fact table
fact_with_card =fact_with_customer.alias("f").join(dim_cards.select("customer_id","card_id","card_key").alias("c"),((F.col("f.user_id") == F.col("c.customer_id")) & (F.col("f.card_id") == F.col("c.card_id"))), how = "left").select("f.*",F.col("c.card_key"))


In [ ]:
#Validation with counts
print("count fact_with_card", fact_with_card.count())
print("counting null for card_key", fact_with_card.filter(F.col("card_key").isNull()).count())

In [ ]:
#Adding merchant_key as foreign key in fact_table
fact_with_merchant = fact_with_card.alias("f").join(dim_merchant.select("merchant_id","merchant_city","merchant_state","mcc","merchant_key").alias("m"),((F.col("f.merchant_id")== F.col("m.merchant_id"))&(F.col("f.merchant_city").eqNullSafe(F.col("m.merchant_city")))&(F.col("f.merchant_state").eqNullSafe(F.col("m.merchant_state")))&(F.col("f.mcc").eqNullSafe(F.col("m.mcc")))),how="left").select("f.*",F.col("m.merchant_key"))

In [ ]:
#Validating the merchant_key
print("count merchant",fact_with_merchant.count())
print("count null merchant_key", fact_with_merchant.filter(F.col("merchant_key").isNull()).count())

In [ ]:
#Adding date_key as foreign key in fact_table
fact_with_date = fact_with_merchant.alias("f").join(dim_date.select("date","date_key").alias("d"),F.col("f.transaction_date") == F.col("d.date"), how = "left").select("f.*", F.col("d.date_key"))

In [ ]:
#Validating the daye_key
print("count fact_with_date",fact_with_date.count())
print("count nulls in date_key", fact_with_date.filter(F.col("date_key").isNull()).count())

In [ ]:
fact_with_card.select(fact_with_card.columns).show()
fact_with_merchant.select(fact_with_merchant.columns).show()
fact_with_date.select(fact_with_date.columns).show()

In [ ]:
#Creating final fact_transactions
fact_transactions = fact_with_date.select("transaction_id","customer_key","card_key","merchant_key","date_key","amount","payment_method","transaction_status","fraud_flag")

In [ ]:
print("count fact_transactions", fact_transactions.count())
print("count distinct fact_transactions", fact_transactions.select("transaction_id").distinct().count())
fact_transactions.select(
    F.sum(F.col("customer_key").isNull().cast("int")).alias("null_customer_keys"),
    F.sum(F.col("card_key").isNull().cast("int")).alias("null_card_keys"),
    F.sum(F.col("merchant_key").isNull().cast("int")).alias("null_merchant_keys"),
    F.sum(F.col("date_key").isNull().cast("int")).alias("null_date_keys")
).show()

In [ ]:
spark.sql("SELECT current_catalog(), current_schema()").show(truncate=False)
spark.sql("SHOW SCHEMAS").show(truncate=False)

Saving final Gold tables as Delta Table

In [ ]:
dim_customers.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.default.dim_customers")
dim_cards.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.default.dim_cards")
dim_merchant.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.default.dim_merchant")
dim_date.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.default.dim_date")
fact_transactions.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.default.fact_transactions")


In [ ]:
#Validating counts for each table
print("dim_customers:",
      spark.table("workspace.default.dim_customers").count())

print("dim_date:",
      spark.table("workspace.default.dim_date").count())

print("dim_cards:",
      spark.table("workspace.default.dim_cards").count())

print("dim_merchant:",
      spark.table("workspace.default.dim_merchant").count())

print("fact_transactions:",
      spark.table("workspace.default.fact_transactions").count())